In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("D:\MINI PROJECT\DATASET\movie_tmdb_details.csv")

print(df.shape)
df.head()


<>:4: SyntaxWarning: invalid escape sequence '\M'
<>:4: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Admin\AppData\Local\Temp\ipykernel_22828\129620061.py:4: SyntaxWarning: invalid escape sequence '\M'
  df = pd.read_csv("D:\MINI PROJECT\DATASET\movie_tmdb_details.csv")


(23138, 18)


,movieId,tmdb_id,title,original_title,overview,tagline,genres,keywords,director,cast_top5,runtime,release_date,release_year,popularity,vote_average,vote_count,poster_path,status
0,1,862.0,Toy Story,Toy Story,"Led by Woody, Andy's toys live happily in his ...",The adventure takes off when toys come to life!,"Family, Comedy, Animation, Adventure","rescue, friendship, mission, jealousy, villain...",John Lasseter,"Tom Hanks, Tim Allen, Don Rickles, Jim Varney,...",81.0,1995-11-22,1995.0,17.9054,8.000,19336.0,/uXDfjJbdP4ijW5hWSBrPrlKpxab.jpg,ok
1,2,8844.0,Jumanji,Jumanji,When siblings Judy and Peter discover an encha...,It's a jungle in here.,"Adventure, Fantasy, Family","giant insect, board game, disappearance, jungl...",Joe Johnston,"Robin Williams, Kirsten Dunst, Bradley Pierce,...",104.0,1995-12-15,1995.0,2.6696,7.243,10985.0,/vgpXmVaVyUL7GGiDeiK1mKEKzcX.jpg,ok
2,3,15602.0,Grumpier Old Men,Grumpier Old Men,A family wedding reignites the ancient feud be...,Still Yelling. Still Fighting. Still Ready for...,"Romance, Comedy","fishing, sequel, old man, best friend, wedding...",Howard Deutch,"Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",101.0,1995-12-22,1995.0,1.9051,6.500,410.0,/1FSXpj5e8l4KH6nVFO5SPUeraOt.jpg,ok
3,4,31357.0,Waiting to Exhale,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",Friends are the people who let you be yourself...,"Comedy, Drama, Romance","based on novel or book, single mother, divorce...",Forest Whitaker,"Whitney Houston, Angela Bassett, Loretta Devin...",127.0,1995-12-22,1995.0,2.3907,6.281,180.0,/qJU6rfil5xLVb5HpJsmmfeSK254.jpg,ok
4,5,11862.0,Father of the Bride Part II,Father of the Bride Part II,Just when George Banks has recovered from his ...,Just when his world is back to normal... he's ...,"Comedy, Family","daughter, baby, parent child relationship, mid...",Charles Shyer,"Steve Martin, Diane Keaton, Martin Short, Kimb...",106.0,1995-12-08,1995.0,2.5283,6.272,780.0,/rj4LBtwQ0uGrpBnCELr716Qo3mw.jpg,ok


In [2]:
#Inspect missing value
df.isnull().sum().sort_values(ascending=False)


tagline           7062
keywords          3394
cast_top5          464
poster_path        225
genres             167
director           113
overview            72
runtime             45
original_title      45
title               45
popularity          45
vote_average        45
release_year        45
release_date        45
vote_count          45
movieId              0
tmdb_id              0
status               0
dtype: int64

In [3]:
#Never dropping as to preserve as much movies as possible to eliminate cold start
TEXT_COLS = ["overview", "tagline", "genres", "keywords", "director", "cast_top5"]
NUM_COLS = ["runtime", "popularity", "vote_average", "vote_count"]

# Fill text columns
for col in TEXT_COLS:
    df[col] = df[col].fillna("")

# Fill numeric columns
df["runtime"] = df["runtime"].fillna(df["runtime"].median())
df["vote_average"] = df["vote_average"].fillna(0)
df["vote_count"] = df["vote_count"].fillna(0)
df["popularity"] = df["popularity"].fillna(0)


In [4]:
#Simple text preprocessing
def clean_text(text):
    text = str(text).lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = " ".join(text.split())
    return text

for col in ["overview", "tagline", "keywords", "genres"]:
    df[col] = df[col].apply(clean_text)


In [5]:
df["combined_text"] = (
    df["overview"] + " " +
    df["genres"] + " " +
    df["keywords"]
)


In [6]:
print("Total movies:", len(df))
print("Movies with empty overview:", (df["overview"] == "").sum())
print("Unique genres count:", df["genres"].nunique())


Total movies: 23138
Movies with empty overview: 72
Unique genres count: 3130


In [7]:
FINAL_COLS = [
    "movieId",
    "tmdb_id",
    "title",
    "overview",
    "genres",
    "keywords",
    "director",
    "cast_top5",
    "runtime",
    "release_year",
    "popularity",
    "vote_average",
    "vote_count",
    "poster_path",
    "combined_text"
]

movies_final = df[FINAL_COLS]
movies_final.to_csv("movies_final.csv", index=False)

print("Saved movies_final.csv")


Saved movies_final.csv


In [ ]:
# ================================
# STEP 1: IMPORTS
# ================================
import pandas as pd
import numpy as np
import sys, os

# Adjust path if needed
PROJECT_ROOT = "D:/MINI PROJECT"
sys.path.append(PROJECT_ROOT)

# ================================
# STEP 2: LOAD CONTEXT
# ================================
from backend.loaders import load_all

context = load_all()

# ================================
# STEP 3: ACCESS DATA
# ================================
movies = context["movies"]
ratings = context["ratings"]
embeddings = context["text_embeddings"]

# ================================
# STEP 4: BASIC CHECKS
# ================================
print("Total movies:", len(movies))
print("Unique movieIds:", movies["movieId"].nunique())
print("Embeddings shape:", embeddings.shape)

# ================================
# STEP 5: FIND DUPLICATES
# ================================
duplicates = movies[movies.duplicated(subset="movieId", keep=False)]

print("\nDuplicate rows count:", len(duplicates))

if len(duplicates) > 0:
    print("\nSample duplicates:")
    print(duplicates[["movieId", "title"]].head(10))

# ================================
# STEP 6: CHECK YEAR COLUMN
# ================================
print("\nColumns:", movies.columns)

if "release_year" in movies.columns:
    print("Max year:", movies["release_year"].max())
else:
    print("release_year NOT FOUND")

# ================================
# STEP 7: FINAL CONSISTENCY CHECK
# ================================
if len(movies) != embeddings.shape[0]:
    print("\n❌ PROBLEM: Movies and embeddings mismatch!")
else:
    print("\n✅ Movies and embeddings aligned")